In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

import LMCE.cfusdlog as cfusdlog
from data_standardization import transform
from LMCE.residual_calculation import residual
from LMCE.error_estimation import error_calculator

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times']

In [3]:
# TABLE_DESCRIPTION = "Comparison of average deviation from desired flight path (meters) of the quadrotor carrying a payload."
TABLE_DESCRIPTION = "Comparison of average deviation from desired flight path (meters) of the quadrotor with no payload."
LABEL = "results_nopayload"

NN_KEY = "nn_output.f_x"
INDI_KEY = "ctrlLee.tau_rpm_fx"

In [4]:
tests = {
    "Circle": {
        "folder": "../data/results/single/figure8",
    },
}

error_na = np.array([])
error_in = np.array([])
for k, v in tests.items():
    errors = [[] for _ in range(4)]
    total_flights = [0 for _ in range(4)]

    methods = {}
    for method in os.listdir(v["folder"]):
        print(method)
        # if method != "na-indi-plAcc" and method != "il-ndi" and method != "indi-plAcc" and method != "lee":
        # # if method != "indi-plAcc" and method != "new_na-indi" and method != "na-indi2":
            # continue
        folder = os.path.join(v["folder"], method)
        for filename in sorted(os.listdir(folder)):
        
            if not (method in methods.keys()):
                methods[method] = []
            
            file_path = os.path.join(folder, filename)

            try:
                data = cfusdlog.decode(file_path)['fixedFrequency']
            except:
                print(f"Failed to read: {filename}")

            if "payload" in v["folder"]:
                real_pos = np.array([data["stateEstimateZ.px"], data["stateEstimateZ.py"], data["stateEstimateZ.pz"]])/1000
                target_pos = np.array([data["ctrltargetZ.x"], data["ctrltargetZ.y"], data["ctrltargetZ.z"]])/1000
            else:
                real_pos = np.array([data["stateEstimateZ.x"], data["stateEstimateZ.y"], data["stateEstimateZ.z"]])/1000
                target_pos = np.array([data["ctrltargetZ.x"], data["ctrltargetZ.y"], data["ctrltargetZ.z"]])/1000
            # plt.plot(data["ctrlLeeP.a_imu_fx"] - data["ctrlLeeP.a_rpm_fx"], label="indi")
            # plt.plot(data["nn_output.f_x"], label="nn")
            # plt.legend()
            # plt.show()
            
            if method == "na-indi" and not error_na.shape[0]:
                error_na = np.linalg.norm(real_pos-target_pos, axis=0)
            if method == "indi" and not error_in.shape[0]:
                error_in = np.linalg.norm(real_pos-target_pos, axis=0)
            
            # print(data["ctrlLeeP.a_imu_fx"][:100])
            # data = transform(data)
            # f_a, tau_a = residual(data)
            # plt.plot((data["ctrlLeeP.tau_imu_fz"] - data["ctrlLeeP.tau_rpm_fz"]), label="online")
            # plt.plot(data["nn_output.f_x"], label="nn")
            # # plt.plot(f_a[:, 2], label="offline")
            # plt.legend()
            # plt.show()
            # real_pos = [data["stateEstimateZ.px"]/1000, data["stateEstimateZ.py"]/1000, data["stateEstimateZ.pz"]/1000]
            # target_pos = np.array([data["ctrltargetZ.x"], data["ctrltargetZ.y"], data["ctrltargetZ.z"]])/1000
            # real_pos = [data["ctrlLeeP.plVelx"], data["ctrlLeeP.plVely"], data["ctrlLeeP.plVelz"]]
            # target_pos = np.array([data["ctrltargetZ.vx"], data["ctrltargetZ.vy"], data["ctrltargetZ.vz"]])/1000
            target_pos = target_pos.tolist()
        
            start_time = data['timestamp'][0]
            start_cutoff = int(((data['timestamp'] - start_time) / 1e3)[0])
            error = error_calculator(real_pos, target_pos, vis=False)
            print(f"{filename}: {error}")

            methods[method].append(float(error))

        print("-"*16)

# plt.plot(error_in, label="indi")
# plt.plot(error_na, label="na-indi")
# plt.legend()
# plt.show()
for k, v in methods.items():
    v = np.array(v)
    print(f"{k} ({len(v)}): {v.mean():.4f} \pm {v.std(ddof=1):.4f}, values={v.tolist()}")

il-ndi
naindi_00: 0.054819683190439686
naindi_01: 0.042563182481666496
naindi_02: 0.040508075106449944
naindi_03: 0.043550235420701064
naindi_04: 0.043923016169967306
naindi_05: 0.0426707285527461
naindi_06: 0.042978639543196785
naindi_07: 0.04600431217781557
naindi_08: 0.04431320988225847
se: 0.04034967446704001
----------------
lee
naindi_00: 0.06882838940071155
naindi_01: 0.06934450999931277
naindi_02: 0.06774928117251214
naindi_027: 0.06843942395050202
naindi_03: 0.06984228199662125
naindi_04: 0.07189929859238989
naindi_05: 0.07269969834169517
naindi_06: 0.069009529364332
naindi_07: 0.07115561801966407
rerte: 0.07177290031901222
----------------
indi
naindi_03: 0.0516792214952914
naindi_04: 0.05447435056339874
naindi_05: 0.033444084582380545
naindi_06: 0.03225224056931642
naindi_07: 0.052785371328651146
naindi_07s: 0.03223513878133393
naindi_08: 0.05407802722610541
naindi_09: 0.03299147950542171
naindi_10: 0.034652091974375736
naindi_11: 0.03595591981467069
----------------
na-indi